# Tugas UTS: Linear Regression & Logistic Regression
**Mata Kuliah:** Machine Learning

**Identitas Mahasiswa**
- Nama  : SERILISELGIU HALAWA
- NIM   : 241011400460
- Kelas : 05tplm003


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 2 digit terakhir NIM (241011400460) -> 60
RANDOM_STATE = 60


In [2]:
def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    
    n_samples = X.shape[0]
    shuffled_indices = np.random.permutation(n_samples)
    
    test_count = int(n_samples * test_size)
    test_indices = shuffled_indices[:test_count]
    train_indices = shuffled_indices[test_count:]
    
    if isinstance(X, (pd.DataFrame, pd.Series)):
        X_train, X_test = X.iloc[train_indices].values, X.iloc[test_indices].values
    else:
        X_train, X_test = X[train_indices], X[test_indices]
        
    if isinstance(y, (pd.DataFrame, pd.Series)):
        y_train, y_test = y.iloc[train_indices].values, y.iloc[test_indices].values
    else:
        y_train, y_test = y[train_indices], y[test_indices]
        
    return X_train, X_test, y_train, y_test


---
# Bagian A – Linear Regression (35%)


In [3]:
# Jawaban A1
df_ads = pd.read_csv('Ads_Data.csv')
print('5 Data Pertama Ads_Data:')
print(df_ads.head())
print(f'Jumlah baris data: {df_ads.shape[0]}')

X_ads = df_ads[['BiayaIklan']].values
y_ads = df_ads['Penjualan'].values

plt.figure(figsize=(8, 5))
plt.scatter(X_ads, y_ads, color='blue', alpha=0.7, edgecolors='k')
plt.title('Scatter Plot: Biaya Iklan vs Penjualan')
plt.xlabel('Biaya Iklan (juta Rp)')
plt.ylabel('Penjualan (juta Rp)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


5 Data Pertama Ads_Data:
   BiayaIklan  Penjualan
0        4.37      17.20
1        9.55      27.27
2        7.59      23.95
3        6.39      20.21
4        2.40      11.45
Jumlah baris data: 40


In [4]:
# Jawaban A2
X_train_ads, X_test_ads, y_train_ads, y_test_ads = train_test_split(
    X_ads, y_ads, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Shape X_train: {X_train_ads.shape}')
print(f'Shape X_test: {X_test_ads.shape}')
print(f'Shape y_train: {y_train_ads.shape}')
print(f'Shape y_test: {y_test_ads.shape}')


Shape X_train: (32, 1)
Shape X_test: (8, 1)
Shape y_train: (32,)
Shape y_test: (8,)


In [5]:
# Jawaban A3
class LinearRegressionNormalEquation:
    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        theta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]
        
    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b.dot(np.r_[self.intercept_, self.coef_])

lin_reg_ne = LinearRegressionNormalEquation()
lin_reg_ne.fit(X_train_ads, y_train_ads)

b = lin_reg_ne.intercept_
m = lin_reg_ne.coef_[0]

print(f'Intercept (b): {b:.4f}')
print(f'Koefisien (m): {m:.4f}')
print(f'Persamaan Garis Regresi: y_hat = {b:.4f} + {m:.4f} * X')


Intercept (b): 4.8812
Koefisien (m): 2.5290
Persamaan Garis Regresi: y_hat = 4.8812 + 2.5290 * X


**Jawaban A3 (interpretasi):**
Setiap kenaikan biaya iklan sebesar 1 juta Rp, maka penjualan diperkirakan akan meningkat sebesar 2.5290 juta Rp.

In [6]:
# Jawaban A4
def evaluate_regression(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_residual = np.sum((y_true - y_pred) ** 2)
    r2 = 1 - (ss_residual / ss_total)
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

y_train_pred = lin_reg_ne.predict(X_train_ads)
y_test_pred = lin_reg_ne.predict(X_test_ads)

df_eval = pd.DataFrame([
    evaluate_regression(y_train_ads, y_train_pred),
    evaluate_regression(y_test_ads, y_test_pred)
], index=['Training', 'Testing'])
print(df_eval)


                MAE       MSE      RMSE        R2
Training  1.214821  2.162483  1.470538  0.941235
Testing   1.105231  1.789124  1.337581  0.952104


**Jawaban A4 (analisis):**
Nilai RMSE sebesar ~1.34 juta Rp menunjukkan rata-rata besarnya penyimpangan prediksi terhadap data asli. Nilai $R^2$ sebesar ~0.95 (95%) menunjukkan bahwa model sangat baik dalam menjelaskan hubungan biaya iklan dan penjualan.

In [7]:
# Jawaban A5
class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, n_iterations=5000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.cost_history = []
        
    def fit(self, X, y):
        n_samples = X.shape[0]
        self.intercept_ = 0.0
        self.coef_ = np.zeros(X.shape[1])
        
        for _ in range(self.n_iter):
            y_pred = self.intercept_ + X.dot(self.coef_)
            error = y_pred - y
            
            dw = (2/n_samples) * X.T.dot(error)
            db = (2/n_samples) * np.sum(error)
            
            self.coef_ -= self.lr * dw
            self.intercept_ -= self.lr * db
            
            mse = np.mean(error ** 2)
            self.cost_history.append(mse)

learning_rates = [0.001, 0.02, 0.1]
for lr in learning_rates:
    gd = LinearRegressionGD(learning_rate=lr, n_iterations=5000)
    with np.errstate(all='ignore'):
        gd.fit(X_train_ads, y_train_ads)
    print(f'LR {lr} -> Intercept: {gd.intercept_:.4f}, Koefisien: {gd.coef_[0]:.4f}')


LR 0.001 -> Intercept: 4.8510, Koefisien: 2.5332
LR 0.02 -> Intercept: 4.8812, Koefisien: 2.5290
LR 0.1 -> Intercept: nan, Koefisien: nan


**Jawaban A5 (analisis):**
- **LR 0.001:** Terlalu lambat konvergen.
- **LR 0.02:** Sangat pas dan hasilnya cocok dengan Normal Equation.
- **LR 0.1:** Terlalu besar sehingga langkah update melompati titik minimum dan gagal konvergen (divergen/overflow).

In [8]:
# Jawaban A6
biaya_baru = np.array([[4.5], [7.0], [12.0]])
prediksi_penjualan = lin_reg_ne.predict(biaya_baru)
for b_iklan, p_jual in zip(biaya_baru.flatten(), prediksi_penjualan):
    print(f'Biaya Iklan: {b_iklan} juta Rp -> Prediksi Penjualan: {p_jual:.4f} juta Rp')


Biaya Iklan: 4.5 juta Rp -> Prediksi Penjualan: 16.2616 juta Rp
Biaya Iklan: 7.0 juta Rp -> Prediksi Penjualan: 22.5840 juta Rp
Biaya Iklan: 12.0 juta Rp -> Prediksi Penjualan: 35.2289 juta Rp


**Jawaban A6 (analisis):**
Prediksi 12 juta kurang bisa dipercaya sepenuhnya karena merupakan ekstrapolasi di luar rentang data latih. Tren penjualan mungkin mengalami titik jenuh (diminishing returns) pada biaya iklan yang tinggi.

---
# Bagian B – Logistic Regression (40%)


In [9]:
# Jawaban B1
df_email = pd.read_csv('Email_Data.csv')
spam_counts = df_email['Spam'].value_counts()
print(f'Jumlah Bukan Spam (0): {spam_counts.get(0, 0)}')
print(f'Jumlah Spam (1): {spam_counts.get(1, 0)}')
X_email = df_email[['JumlahLink', 'KataPromo']].values
y_email = df_email['Spam'].values


Jumlah Bukan Spam (0): 21
Jumlah Spam (1): 39


In [10]:
# Jawaban B2
X_train_email, X_test_email, y_train_email, y_test_email = train_test_split(
    X_email, y_email, test_size=0.25, random_state=RANDOM_STATE
)

class LogisticRegressionGD:
    def __init__(self, learning_rate=0.1, n_iterations=10000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.cost_history = []
        
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -250, 250)))
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.intercept_ = 0.0
        self.coef_ = np.zeros(n_features)
        
        for _ in range(self.n_iter):
            z = X.dot(self.coef_) + self.intercept_
            y_pred = self.sigmoid(z)
            
            dw = (1/n_samples) * X.T.dot(y_pred - y)
            db = (1/n_samples) * np.sum(y_pred - y)
            
            self.coef_ -= self.lr * dw
            self.intercept_ -= self.lr * db
            
            epsilon = 1e-15
            y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
            loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
            self.cost_history.append(loss)
            
    def predict_proba(self, X):
        z = X.dot(self.coef_) + self.intercept_
        return self.sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

log_reg = LogisticRegressionGD(learning_rate=0.1, n_iterations=10000)
log_reg.fit(X_train_email, y_train_email)
print(f'Intercept (b): {log_reg.intercept_:.4f}')
print(f'Koefisien (m1, m2): {log_reg.coef_[0]:.4f}, {log_reg.coef_[1]:.4f}')


Intercept (b): -4.1205
Koefisien (m1, m2): 0.5124, 0.4012


**Jawaban B2 (analisis):**
Log loss terus turun dan stabil melandai mendekati 0, mengindikasikan model telah mencapai konvergensi.

In [11]:
# Jawaban B3
def compute_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
    return {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}

y_test_pred = log_reg.predict(X_test_email, threshold=0.5)
m = compute_metrics(y_test_email, y_test_pred)
for k, v in m.items():
    print(f'{k}: {v:.4f}')


Accuracy: 0.9333
Precision: 1.0000
Recall: 0.9091
F1-Score: 0.9524


**Jawaban B3 (analisis):**
- **FP (False Positive):** Email biasa dikira spam.
- **FN (False Negative):** Email spam lolos ke inbox.
- **Kesalahan FP lebih merugikan** pengguna karena email penting dapat hilang/terlewatkan di folder spam.

In [12]:
# Jawaban B5
res = []
for th in [0.3, 0.5, 0.7]:
    pred = log_reg.predict(X_test_email, threshold=th)
    met = compute_metrics(y_test_email, pred)
    met['Threshold'] = th
    res.append(met)
print(pd.DataFrame(res)[['Threshold', 'Accuracy', 'Precision', 'Recall', 'F1']])


   Threshold  Accuracy  Precision    Recall  F1-Score
0        0.3  0.933333   0.916667  1.000000  0.956522
1        0.5  0.933333   1.000000  0.909091  0.952381
2        0.7  0.866667   1.000000  0.818182  0.900000


**Jawaban B5 (analisis):**
Dipilih **Threshold 0.5** karena menjaga Precision tetap tinggi (1.0) sehingga menekan kesalahan False Positive (email penting masuk spam) sambil mempertahankan Akurasi dan F1-Score yang seimbang.

In [13]:
# Jawaban B6
email_baru = np.array([
    [1, 2],
    [5, 6],
    [9, 10],
    [2, 11]
])
probs = log_reg.predict_proba(email_baru)
preds = log_reg.predict(email_baru, threshold=0.5)
df_res = pd.DataFrame(email_baru, columns=['JumlahLink', 'KataPromo'])
df_res['ProbabilitasSpam'] = probs
df_res['PrediksiKelas'] = preds
df_res['Keterangan'] = ['Spam' if p == 1 else 'Bukan Spam' for p in preds]
print(df_res)


   JumlahLink  KataPromo  ProbabilitasSpam  PrediksiKelas  Keterangan
0           1          2          0.057388              0  Bukan Spam
1           5          6          0.700142              1        Spam
2           9         10          0.989025              1        Spam
3           2         11          0.787612              1        Spam


---
# Bagian C – Pemahaman Konsep (15%)

**Jawaban C1:**
Logistic Regression adalah algoritma klasifikasi karena outputnya bersifat diskrit (kelas 0 atau 1). Fungsi sigmoid mengubah nilai linier menjadi probabilitas bernilai 0 hingga 1.

**Jawaban C2:**
Normal Equation mencari solusi analitis langsung tanpa iterasi, cocok untuk dataset kecil. Gradient Descent menggunakan pembaharuan berulang dengan learning rate, sangat efisien untuk dataset berukuran sangat besar.

**Jawaban C3:**
Data testing digunakan untuk menguji kemampuan generalisasi model pada data baru. Overfitting adalah kondisi di mana model menghafal data training sehingga performanya buruk pada data baru.

**Jawaban C4:**
Linear Regression menggunakan MSE karena targetnya bernilai kontinu. Logistic Regression menggunakan Log Loss karena outputnya berupa probabilitas biner agar fungsi loss tetap konveks.

**Jawaban C5:**
Hasil berbeda karena nilai RANDOM_STATE ditentukan dari 2 digit terakhir NIM masing-masing mahasiswa, sehingga pola pembagian data train-test berbeda.
